# Used Car Price Prediction – D'CODE AI/ML Round 2
Regression: predict `price` from car features. Flow: EDA → cleaning/feature engineering → baseline + tree models → validation → predictions.

In [ ]:
!pip install -q catboost --break-system-packages
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

# Upload train.csv, test.csv, sample_submission.csv to Colab (or edit these paths)
train = pd.read_csv(r"C:\Users\coolh\OneDrive\Documents\DCODE STUFF\train(1).csv.csv")
test  = pd.read_csv(r"C:\Users\coolh\OneDrive\Documents\DCODE STUFF\test(2).csv.csv")
print(train.shape, test.shape)

## 1. EDA

In [ ]:
train['price'] = train.price.str.replace(r'[\$,]', '', regex=True).astype(float)
train.info()
print(train.isna().sum())
print(train.price.describe())

**Target is extremely right-skewed** (skew ≈ 19, max ≈ $2.95M vs median $31k). Log-transform makes it near-symmetric, so I'll train on `log1p(price)`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(train.price, bins=60, ax=ax[0]); ax[0].set_title('price (skew %.1f)' % train.price.skew())
sns.histplot(np.log1p(train.price), bins=60, ax=ax[1]); ax[1].set_title('log1p(price) (skew %.2f)' % np.log1p(train.price).skew())
plt.show()
print(train.sort_values('price').tail(5)[['brand','model','model_year','milage','price']])

In [ ]:
train['mileage_num'] = train.milage.str.replace(r'[^\d]', '', regex=True).astype(float)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].scatter(train.mileage_num, np.log1p(train.price), s=6, alpha=.4); ax[0].set_title('mileage vs log price')
ax[1].scatter(train.model_year, np.log1p(train.price), s=6, alpha=.4); ax[1].set_title('model year vs log price')
train.groupby('brand').price.median().sort_values().tail(12).plot.barh(ax=ax[2]); ax[2].set_title('median price, top brands')
plt.tight_layout(); plt.show()
print(train.fuel_type.value_counts(dropna=False)); print(train.clean_title.value_counts(dropna=False))
print('unique brand/model:', train.brand.nunique(), train.model.nunique())

Observations: mileage lowers price, newer year raises it, luxury/exotic brands dominate the top. `model` has ~1,700 unique values (high cardinality). `fuel_type` contains junk values ('–', 'not supported'). `clean_title` is either 'Yes' or blank — I treat blank as its own signal (no clean title on record), not as a random gap.

## 2. Cleaning & feature engineering

In [ ]:
def make_features(d):
    d = d.copy()
    d['mileage'] = d.milage.str.replace(r'[^\d]', '', regex=True).astype(float)
    d['car_age'] = 2026 - d.model_year
    d['mi_per_year'] = d.mileage / d.car_age.clip(lower=1)
    d['fuel_type'] = d.fuel_type.replace({'–': 'Unknown', 'not supported': 'Unknown'}).fillna('Unknown')
    d['acc_missing'] = d.accident.isna().astype(int)              # missing itself may carry meaning
    d['accident'] = d.accident.map({'None reported': 0, 'At least 1 accident or damage reported': 1})
    d['clean_title'] = d.clean_title.notna().astype(int)         # blank -> 0
    d['disp_missing'] = d.engine_displacement_l.isna().astype(int)
    d['cyl_missing'] = d.engine_cylinders.isna().astype(int)     # typically EVs / unlisted engines
    d['brand_model'] = d.brand + ' ' + d.model
    d['model_first'] = d.brand + ' ' + d.model.str.split().str[0]  # coarser model group
    d['ext_col'] = d.ext_col.str.lower(); d['int_col'] = d.int_col.str.lower()
    t = d.transmission_type.str.lower()
    d['trans'] = np.where(t.str.contains('manual'), 'manual', np.where(t.str.contains('cvt'), 'cvt', 'auto'))
    return d

cat_cols = ['brand','model','fuel_type','ext_col','int_col','trans','brand_model','model_first','transmission_type']
num_cols = ['model_year','mileage','car_age','accident','clean_title','engine_displacement_l','engine_cylinders',
            'acc_missing','disp_missing','cyl_missing','mi_per_year']
feats = cat_cols + num_cols
X = make_features(train)[feats]; y = np.log1p(train.price)
X_test = make_features(test)[feats]
X.shape, X_test.shape

CatBoost handles the high-cardinality categoricals (`model`, `brand_model`) natively with ordered target statistics, so no manual target-encoding leakage is needed; missing numerics are left as NaN for it. For Ridge/RF I impute with the train median only.

## 3–4. Models & validation (80/20 split)

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42)
results = {}
def report(name, pred_log):
    p, a = np.expm1(pred_log), np.expm1(y_va)
    results[name] = dict(RMSE=mean_squared_error(a, p)**.5, MAE=mean_absolute_error(a, p),
                         logRMSE=mean_squared_error(y_va, pred_log)**.5)
    print(name, {k: round(v, 3) for k, v in results[name].items()})

# --- Baseline: Ridge (linear) ---
def prep_lin(A, ref):
    A = A.copy()
    for c in ['engine_displacement_l', 'engine_cylinders']: A[c] = A[c].fillna(ref[c].median())
    A['accident'] = A.accident.fillna(0)
    A['mileage'] = np.log1p(A.mileage); A['mi_per_year'] = np.log1p(A.mi_per_year)
    return A
lin_cat = ['brand','fuel_type','trans','transmission_type','model_first']
ct = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore', min_frequency=3), lin_cat),
                        ('n', StandardScaler(), num_cols)])   # fit on train only (inside pipeline)
ridge = make_pipeline(ct, RidgeCV(alphas=np.logspace(-1, 2, 10))).fit(prep_lin(X_tr, X_tr), y_tr)
report('Ridge baseline', ridge.predict(prep_lin(X_va, X_tr)))

# --- Random Forest ---
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
def prep_rf(A, fit=False):
    B = A.copy(); B[cat_cols] = oe.fit_transform(A[cat_cols]) if fit else oe.transform(A[cat_cols]); return B.fillna(-1)
rf = RandomForestRegressor(500, min_samples_leaf=2, n_jobs=-1, random_state=0).fit(prep_rf(X_tr, True), y_tr)
report('Random Forest', rf.predict(prep_rf(X_va)))

# --- CatBoost ---
cb = CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=6, verbose=0, random_seed=0, cat_features=cat_cols)
cb.fit(X_tr, y_tr); report('CatBoost', cb.predict(X_va))
pd.DataFrame(results).T

**Why these models:** Ridge = simple, interpretable baseline. Random Forest = non-linear, mixed data. CatBoost = gradient boosting with native categorical handling, ideal for the high-cardinality `model` column.

**Reading the numbers:** raw-dollar RMSE is dominated by a few exotic cars (a $2.95M Maserati, $1.95M Bugatti) that no model can predict from 3k rows, so a single split can swing RMSE a lot. I therefore also report log-RMSE (relative error) and 5-fold CV below.

## 5. Cross-validation & evaluation

In [ ]:
kf = KFold(5, shuffle=True, random_state=1)
Xr, yr = X.reset_index(drop=True), y.reset_index(drop=True)
oof = np.zeros(len(Xr))
for a, b in kf.split(Xr):
    m = CatBoostRegressor(iterations=500, learning_rate=0.07, task_type='GPU', depth=6, verbose=100, random_seed=0, cat_features=cat_cols)
    oof[b] = m.fit(Xr.iloc[a], yr.iloc[a]).predict(Xr.iloc[b])
p, act = np.expm1(oof), train.price.values
k = act < 1e6
print('CV RMSE (all)      : %.0f' % mean_squared_error(act, p)**.5)
print('CV RMSE (price<1M) : %.0f' % mean_squared_error(act[k], p[k])**.5)
print('CV MAE             : %.0f' % mean_absolute_error(act, p))
print('CV log-RMSE        : %.4f' % mean_squared_error(yr, oof)**.5)

plt.figure(figsize=(5.5, 5)); plt.scatter(act[k], p[k], s=6, alpha=.4)
plt.plot([0, 3e5], [0, 3e5], 'r--'); plt.xlabel('actual'); plt.ylabel('predicted (CV)'); plt.title('Predicted vs actual'); plt.show()

## 6. Final model & submission
Average of 5 CatBoost seeds trained on all of train.csv (same feature pipeline applied to test).

In [ ]:
preds = np.zeros(len(X_test))
for s in range(5):
    m = CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=6, verbose=0, random_seed=s, cat_features=cat_cols)
    preds += m.fit(X, y).predict(X_test) / 5
sub = pd.DataFrame({'id': test.id, 'price': np.expm1(preds).round(0)})
sub.to_csv('submission.csv', index=False)
print(sub.shape, sub.price.describe()); sub.head()